In [1]:
%%writefile app.py
import os
from haystack import Pipeline
from haystack_integrations.document_stores.elasticsearch import ElasticsearchDocumentStore
from haystack_integrations.components.retrievers.elasticsearch import ElasticsearchBM25Retriever
from haystack.components.builders.prompt_builder import PromptBuilder
from haystack.components.generators import HuggingFaceAPIGenerator
from haystack.components.converters import TextFileToDocument
from haystack.components.writers import DocumentWriter
from haystack.components.preprocessors import DocumentSplitter
from haystack.utils import Secret
from elasticsearch import Elasticsearch, NotFoundError
import huggingface_hub

username = "elastic"
password = "ma6BKlMY"
index_name = "course-transcripts"

document_store = ElasticsearchDocumentStore(
    hosts="http://localhost:9200",
    basic_auth=(username, password),
    index=index_name
)

## Below code was commented out after first run to avoid index document duplication. 

#converter = TextFileToDocument()
#splitter = DocumentSplitter()
#writer = DocumentWriter(document_store, update_existing_documents=True)

#indexing_pipeline = Pipeline()
#indexing_pipeline.add_component("converter", converter)
#indexing_pipeline.add_component("splitter", splitter)
#indexing_pipeline.add_component("writer", writer)

#indexing_pipeline.connect("converter", "splitter")
#indexing_pipeline.connect("splitter", "writer") 

#import glob
#files = sorted(glob.glob("transcripts/transcripts/*.txt"))

#indexing_pipeline.run({"converter": {"sources": files}})

template = """
  <|system|>
  You are a helpful assistant.<|end|>
  <|user|>
  Given the following information, answer the question.

  Context: 
  {% for document in documents %}
      {{ document.content }}
  {% endfor %}

  Question: {{ query }}?<|end|>
  <|assistant|>"""

with open("huggingface_token", "r") as FH:
    data = FH.read()
    huggingface_token = data.strip()

from huggingface_hub import login
login(token=huggingface_token)

retriever = ElasticsearchBM25Retriever(document_store=document_store)
generator = HuggingFaceAPIGenerator(
    api_type="serverless_inference_api",
    api_params={"model": "microsoft/Phi-3.5-mini-instruct", "max_new_tokens": 200 })

rag_pipeline = Pipeline()
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.add_component("prompt_builder", PromptBuilder(template=template))
rag_pipeline.add_component("llm", generator)
rag_pipeline.connect("retriever", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder", "llm")

import streamlit as st
st.title("💬 Course Chatbot")
st.caption("🚀 Interactive Q&A with Elasticsearch, Haystack, and HuggingFace")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    st.chat_message(msg["role"]).write(msg["content"])

if prompt := st.chat_input("Ask a question about the course transcripts"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)

    response = rag_pipeline.run({
        "prompt_builder": {"query": prompt},
        "retriever": {"query": prompt}
    })['llm']['replies'][0]

    st.session_state.messages.append({"role": "assistant", "content": response})
    st.chat_message("assistant").write(response)

Overwriting app.py


In [2]:
!curl https://loca.lt/mytunnelpassword

34.60.116.115

### Q3.3 - Comparing RAG vs Fine-Tuned LoRA LLM ###

The RAG pipeline consistently generated more accurate responses because it retrieved specific lecture transcripts from Elasticsearch before generation. This ensured that the answers were not only relevant but also directly traceable to course material. In contrast, the fine-tuned LLM tended to produce more generic or occasionally hallucinated content, especially when asked questions that were not explicitly seen during fine-tuning. This is because the transcripts were applied to the weighting system in the beginning rather than being retrieved from Elasticsearch at query time. The LoRA model did capture the tone and general knowledge of the course but lacked the ability to pull recent and precise content.

For example, we asked both models "Why might Elasticsearch be preferred over MongoDB?". Generally comparing the two responses:

The Fine-Tuned LoRA model highlighted the general differences between the two well, but was not very technical in-depth. It briefly described some tradeoffs between the two, but did not provide any example scenarios where Elasticsearch might be the desired engine. 

The RAG model used more technical terms in its explanation, provided more reasoning behind WHEN and WHY Elastichsearch could be better in certain situations, and gave real-life examples (probably straight from lecture transcripts) to support these claims. 